In [10]:
import json, pandas as pd

with open('../data/raw/pubchem_assay_details.json') as f:
    assay_data = json.load(f)

rows = []
for assay in assay_data:
    try:
        pc_assay = assay['PC_AssaySubmit']['assay']['descr']
        aid  = pc_assay.get('aid', {}).get('id', 'N/A')
        name = pc_assay.get('name', 'N/A')
        desc = pc_assay.get('description', [''])[0][:120]
        rows.append({'AID': aid, 'name': name, 'description': desc})
    except (KeyError, TypeError):
        continue

assay_meta = pd.DataFrame(rows)
print(assay_meta)
pd.set_option('display.max_colwidth', None)
print(assay_meta[['AID', 'name']])
assay_meta.to_csv('../data/processed/pcsk9_assay_metadata.csv', index=False)

      AID                                               name  \
0  208829  Compound was tested for the inhibition of subt...   
1  208830             Inhibitory activity against subtilisin   
2  208831  Ratio of the kinetic constants (ki X 10e3) ver...   
3  208832  Ratio of the kinetic constants kH versus ki wa...   
4  208833  The ratio (ki + kH)/ki (turnover ratio) was de...   
5  208834             Inhibitory activity against subtilisin   
6  208835             Inhibitory activity against subtilisin   
7  208836             Inhibitory activity against subtilisin   

                                         description  
0  Compound was tested for the inhibition of subt...  
1  Title: O'-(epoxyalkyl)tyrosines and (epoxyalky...  
2  Title: O'-(epoxyalkyl)tyrosines and (epoxyalky...  
3  Title: O'-(epoxyalkyl)tyrosines and (epoxyalky...  
4  Title: O'-(epoxyalkyl)tyrosines and (epoxyalky...  
5  Title: O'-(epoxyalkyl)tyrosines and (epoxyalky...  
6  Title: O'-(epoxyalkyl)tyrosines and

In [17]:
import requests, time

AID = 1542895   # Replace with an AID from your assay_meta table

url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{AID}/concise/JSON'
r = requests.get(url)
data = r.json()

# Parse the concise table into a DataFrame
table = data['Table']
columns = table['Columns']['Column']  # already the list of column name strings

rows = [row['Cell'] for row in table['Row']]

activity_df = pd.DataFrame(rows, columns=columns)
activity_df.to_csv(f'../data/processed/assay_{AID}_activity.csv', index=False)
print(f'{len(activity_df)} compound activity records fetched')
activity_df.head()

2 compound activity records fetched


,AID,SID,CID,Activity Outcome,Target Accession,Target GeneID,Activity Value [uM],Activity Name,Assay Name,Assay Type,PubMed ID,RNAi
0,1542895,440117162,155515410,Unspecified,P01130,3949,,,Binding affinity to C-terminal His tagged purified recombinant PCSK9 D374Y mutant (unknown origin) expressed in HEK293 cells assessed as reduction in PCSK9/LDLR protein-protein interaction at 100 uM by chemiluminescent assay,Other,30996774,
1,1542895,440117162,155515410,Unspecified,Q8NBP7,255738,,,Binding affinity to C-terminal His tagged purified recombinant PCSK9 D374Y mutant (unknown origin) expressed in HEK293 cells assessed as reduction in PCSK9/LDLR protein-protein interaction at 100 uM by chemiluminescent assay,Other,30996774,


In [12]:
url = 'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/target/genesymbol/PCSK9/aids/JSON'
response = requests.get(url)
aids = response.json()['IdentifierList']['AID']
print(f'Total PCSK9-related assays available: {len(aids)}')

Total PCSK9-related assays available: 142


In [13]:
print(f'Total PCSK9-related assays available: {len(aids)}')

# Pull more assay details, e.g. the next 40
import time
more_assay_data = []
for aid in aids[10:50]:
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/JSON'
    r = requests.get(url)
    if r.status_code == 200:
        more_assay_data.append(r.json())
    time.sleep(0.5)

print(f'Fetched {len(more_assay_data)} more assay records')

Total PCSK9-related assays available: 142
Fetched 27 more assay records


In [15]:
import time
more_assay_data = []
for aid in aids[10:50]:
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{aid}/JSON'
    r = requests.get(url)
    if r.status_code == 200:
        more_assay_data.append(r.json())
    time.sleep(0.5)

print(f'Fetched {len(more_assay_data)} more assay records')

Fetched 27 more assay records


In [16]:
all_assay_data = assay_data + more_assay_data

rows = []
for assay in all_assay_data:
    try:
        pc_assay = assay['PC_AssaySubmit']['assay']['descr']
        aid  = pc_assay.get('aid', {}).get('id', 'N/A')
        name = pc_assay.get('name', 'N/A')
        desc = pc_assay.get('description', [''])[0][:200]
        rows.append({'AID': aid, 'name': name, 'description': desc})
    except (KeyError, TypeError):
        continue

assay_meta_full = pd.DataFrame(rows)
pcsk9_specific = assay_meta_full[assay_meta_full['name'].str.contains('PCSK9', case=False, na=False) | 
                                   assay_meta_full['description'].str.contains('PCSK9', case=False, na=False)]
print(pcsk9_specific[['AID', 'name']])

        AID  \
9   1390359   
10  1390360   
11  1390361   
12  1390362   
13  1390363   
14  1390364   
15  1390365   
16  1390366   
17  1390367   
18  1390368   
19  1390369   
20  1390378   
21  1390379   
22  1390380   
23  1390381   
24  1390382   
25  1390383   
26  1390385   
27  1519252   
28  1542895   
29  1542897   
30  1542898   
31  1542899   
32  1542900   
33  1542901   
34  1563568   

                                                                                                                                                                                                                                                                                                                       name  
9                                                                                                                                                                                                                                                                           Inhibiti

In [18]:
AID = 1542897
url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/assay/aid/{AID}/concise/JSON'
r = requests.get(url)
data = r.json()

table = data['Table']
columns = table['Columns']['Column']
rows = [row['Cell'] for row in table['Row']]

activity_df2 = pd.DataFrame(rows, columns=columns)
print(f'{len(activity_df2)} compound activity records fetched')
print(activity_df2['Activity Outcome'].value_counts())
activity_df2.head()

2 compound activity records fetched
Activity Outcome
Unspecified    2
Name: count, dtype: int64


,AID,SID,CID,Activity Outcome,Target Accession,Target GeneID,Activity Value [uM],Activity Name,Assay Name,Assay Type,PubMed ID,RNAi
0,1542897,440206803,155560422,Unspecified,P01130,3949,,,Inhibition of C-terminal His tagged purified recombinant PCSK9 D374Y mutant (unknown origin) and LDLR protein-protein interaction expressed in human HepG2 cells assessed as LDL-C uptake level at 10 uM incubated for 2 hrs by fluorescent LDL uptake cell-based assay (42.1 +/- 1.6%),Other,30996774,
1,1542897,440206803,155560422,Unspecified,Q8NBP7,255738,,,Inhibition of C-terminal His tagged purified recombinant PCSK9 D374Y mutant (unknown origin) and LDLR protein-protein interaction expressed in human HepG2 cells assessed as LDL-C uptake level at 10 uM incubated for 2 hrs by fluorescent LDL uptake cell-based assay (42.1 +/- 1.6%),Other,30996774,


In [19]:
import pandas as pd

variant_activity = pd.DataFrame([
    {'variant': 'D374Y', 'gene': 'PCSK9', 'type': 'GOF',
     'domain': 'Catalytic', 'pubchem_aid': 1542895,
     'assay_readout': 'PCSK9/LDLR protein-protein interaction (chemiluminescent)',
     'functional_effect': 'Increased PCSK9-LDLR binding, reduced LDL uptake',
     'pubmed_id': 30996774, 'fh_associated': True},
    {'variant': 'S127R', 'gene': 'PCSK9', 'type': 'GOF',
     'domain': 'Prodomain', 'pubchem_aid': None,
     'assay_readout': 'Literature-confirmed GOF',
     'functional_effect': 'Increased PCSK9 secretion and activity',
     'pubmed_id': None, 'fh_associated': True},
    {'variant': 'F216L', 'gene': 'PCSK9', 'type': 'GOF',
     'domain': 'Catalytic', 'pubchem_aid': None,
     'assay_readout': 'Literature-confirmed GOF',
     'functional_effect': 'Increased PCSK9 activity',
     'pubmed_id': None, 'fh_associated': True},
    {'variant': 'R218S', 'gene': 'PCSK9', 'type': 'GOF',
     'domain': 'Catalytic', 'pubchem_aid': None,
     'assay_readout': 'Literature-confirmed GOF',
     'functional_effect': 'Increased PCSK9 activity',
     'pubmed_id': None, 'fh_associated': True},
])

variant_activity.to_csv('../data/processed/variant_activity_table.csv', index=False)
print(variant_activity)

  variant   gene type     domain  pubchem_aid  \
0   D374Y  PCSK9  GOF  Catalytic    1542895.0   
1   S127R  PCSK9  GOF  Prodomain          NaN   
2   F216L  PCSK9  GOF  Catalytic          NaN   
3   R218S  PCSK9  GOF  Catalytic          NaN   

                                               assay_readout  \
0  PCSK9/LDLR protein-protein interaction (chemiluminescent)   
1                                   Literature-confirmed GOF   
2                                   Literature-confirmed GOF   
3                                   Literature-confirmed GOF   

                                  functional_effect   pubmed_id  fh_associated  
0  Increased PCSK9-LDLR binding, reduced LDL uptake  30996774.0           True  
1            Increased PCSK9 secretion and activity         NaN           True  
2                          Increased PCSK9 activity         NaN           True  
3                          Increased PCSK9 activity         NaN           True  
